# Notebook 00 — Framework Overview and Data Flow

Welcome to the **PhosCrosstalk** educational notebook suite.

PhosCrosstalk is a systems-level ODE modelling framework for phosphoproteomic  
time-series data.  It combines:

* **Phosphosite kinetics** — per-site dephosphorylation (k_off) and kinase-driven activation
* **Protein abundance dynamics** — synthesis, degradation, and crosstalk-modulated production
* **mRNA regulation** — transcription-factor network weighting of protein synthesis
* **Network priors** — PTM-database crosstalk (Cg/Cl) and kinase-substrate weights (K_site_kin)

All parameters are estimated jointly by gradient-based multi-start optimisation.


## Data Flow

```
Raw data (phospho CSV + mRNA CSV + kinase TSV + TF net)
   ↓  data_loader.py
P_data  (N×T)  phosphosite intensities
A_data  (K×T)  protein abundances
rna_matrix (K×T_rna)  mRNA levels
K_site_kin (N×M)  kinase–site weights
Cg / Cl    (N×N)  global / local PTM crosstalk
tf_prot_weights    TF→protein synthesis weights
   ↓  optimization.py / weighting.py
theta ∈ R^(2K+2+3M+N+4)   parameter vector (log-space)
bounds (xl, xu)            biologically constrained
W_data (N×T), W_prot (K×T) observation weights
   ↓  multistarts.py + diffrax ODE
theta_best,  loss = [L_phospho, L_prot, L_mrna, L_reg]
   ↓  analysis.py
Fitted time-series, rate parameters, diagnostics
   ↓  (optional) neuralODE / PINN
Refined rates or universal ODE solution
```


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Load and inspect the sample data files

In [2]:
# Phosphoproteomic data (protein rows + phosphosite rows)
df_pp = pd.read_csv(SAMPLE_DIR / "protephospho.csv")
print("protephospho.csv — shape:", df_pp.shape)
df_pp.head(6)


protephospho.csv — shape: (12, 16)


,GeneID,Psite,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14
0,EGFR,NaN,0.238620,0.661583,0.561625,0.256993,0.218375,0.001000,0.297629,0.679325,1.000000,0.700569,0.342600,0.191126,0.343815,0.493533
1,EGFR,Y1068,0.676981,1.000000,0.986889,0.997060,0.702258,0.611721,0.729335,0.676035,0.513826,0.294383,0.116206,0.001000,0.377601,0.933599
2,EGFR,S1173,0.834016,1.000000,0.971292,0.939213,0.726710,0.551242,0.824882,0.808167,0.617930,0.385087,0.149977,0.001000,0.386786,0.937064
3,EGFR,T992,0.745444,0.954675,0.989117,0.956949,0.745293,0.566586,0.906617,0.916132,0.486997,0.330734,0.094406,0.001000,0.486944,1.000000
4,MET,NaN,0.599926,0.412831,0.168461,0.001000,0.151194,0.403536,0.662757,0.788038,0.585218,0.308136,0.309078,0.501070,0.896752,1.000000
5,MET,Y1068,0.637518,0.807408,0.752220,0.673991,0.529649,0.630519,0.810065,0.784417,0.423224,0.148690,0.098071,0.001000,0.458757,1.000000


In [3]:
# mRNA data
df_rna = pd.read_csv(SAMPLE_DIR / "mrna.csv")
print("mrna.csv — shape:", df_rna.shape)
df_rna.head()


mrna.csv — shape: (3, 15)


,GeneID,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14
0,EGFR,0.508061,0.714054,0.835834,0.547367,0.124301,0.001000,0.384592,0.585721,1.000000,0.897644,0.555600,0.104845,0.583240,0.587438
1,MET,0.562943,0.419958,0.220494,0.001000,0.182170,0.530785,0.798708,0.755601,0.480009,0.322054,0.361790,0.609128,0.849289,1.000000
2,ERBB2,0.207023,0.001000,0.027794,0.272589,0.515897,0.695064,0.552609,0.320322,0.211322,0.335872,0.643669,1.000000,0.913174,0.722216


In [4]:
# Kinase–site weights
df_kin = pd.read_csv(SAMPLE_DIR / "kinase_sites.tsv", sep="\t")
print("kinase_sites.tsv — shape:", df_kin.shape)
df_kin.head()


kinase_sites.tsv — shape: (14, 3)


,Site,Kinase,weight
0,EGFR_Y1068,MET,0.842946
1,EGFR_Y1068,EGFR,0.157054
2,EGFR_S1173,EGFR,0.679489
3,EGFR_S1173,MET,0.320511
4,EGFR_T992,EGFR,1.000000


In [5]:
# TF–mRNA weights
df_tf = pd.read_csv(SAMPLE_DIR / "tf_mrna.csv")
print("tf_mrna.csv — shape:", df_tf.shape)
df_tf.head()


tf_mrna.csv — shape: (4, 3)


,Source,Target,Weight
0,ERBB2,EGFR,0.747773
1,ERBB2,MET,0.693236
2,MET,MET,0.926073
3,MET,ERBB2,0.873950


## Instantiate ModelDims from the sample data

In [6]:
from phoscrosstalk.config import ModelDims
from phoscrosstalk.data_loader import load_site_data, load_kinase_site_matrix

timepoints = list(range(1, 15))   # 14 time points x1..x14

sites, proteins, site_prot_idx, positions, t, Y, A_data, A_proteins = \
    load_site_data(SAMPLE_DIR / "protephospho.csv", timepoints)

K_site_kin, kinases = load_kinase_site_matrix(SAMPLE_DIR / "kinase_sites.tsv", sites)

K = len(proteins)   # number of proteins
M = len(kinases)    # number of kinases
N = len(sites)      # number of phosphosites

dims = ModelDims.set_dims(K, M, N)
print(f"ModelDims  K={dims.K}  M={dims.M}  N={dims.N}")
print(f"  K = {dims.K} proteins  : {proteins}")
print(f"  M = {dims.M} kinases   : {kinases}")
print(f"  N = {dims.N} phosphosites: {sites}")


ModelDims  K=3  M=2  N=9
  K = 3 proteins  : ['EGFR', 'ERBB2', 'MET']
  M = 2 kinases   : ['EGFR', 'MET']
  N = 9 phosphosites: ['EGFR_Y1068', 'EGFR_S1173', 'EGFR_T992', 'MET_Y1068', 'MET_S1173', 'MET_T992', 'ERBB2_Y1068', 'ERBB2_S1173', 'ERBB2_T992']


### Biological meaning of K, M, N

| Symbol | Count | Meaning |
|--------|-------|---------|
| K | 3 | Receptor-tyrosine kinase proteins tracked by abundance |
| M | 2 | Kinases with known substrate-site weights (EGFR, MET) |
| N | 9 | Phosphosites whose intensities are measured over time |

The **theta** vector has dimension `2K + 2 + 3M + N + 4` = **`2·3 + 2 + 3·2 + 9 + 4 = 25`**.


## Module summary

In [7]:
modules = {
    "phoscrosstalk.config":        "ModelDims, load_config, validate_config",
    "phoscrosstalk.data_loader":   "load_site_data, load_rna_data, load_kinase_site_matrix, load_tf_network",
    "phoscrosstalk.weighting":     "build_weight_matrices",
    "phoscrosstalk.optimization":  "create_bounds, build_parameter_labels",
    "phoscrosstalk.mechanisms":    "decode_theta, make_rhs",
    "phoscrosstalk.derived_rates": "make_k_act_fn, make_s_prod_fn",
    "phoscrosstalk.simulation":    "simulate",
}
df_mod = pd.DataFrame(list(modules.items()), columns=["Module", "Key exports"])
print(df_mod.to_string(index=False))


                     Module                                                             Key exports
       phoscrosstalk.config                                 ModelDims, load_config, validate_config
  phoscrosstalk.data_loader load_site_data, load_rna_data, load_kinase_site_matrix, load_tf_network
    phoscrosstalk.weighting                                                   build_weight_matrices
 phoscrosstalk.optimization                                   create_bounds, build_parameter_labels
   phoscrosstalk.mechanisms                                                  decode_theta, make_rhs
phoscrosstalk.derived_rates                                           make_k_act_fn, make_s_prod_fn
   phoscrosstalk.simulation                                                                simulate
